# 4. Model Evaluation

**Projeto:** Classificação de ataques de cibersegurança  
**Modelo selecionado:** Linear SVM (TF-IDF + LinearSVC)

Nesta etapa analisamos os resultados do conjunto de **teste**, separado durante a preparação. Os resultados são lidos dos ficheiros gerados anteriormente, sem voltar a treinar nem a escolher modelos com base no teste.

## 1. Carregar resultados guardados

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

candidates = [Path("models"), Path("../models")]
models_dir = next((p for p in candidates if (p / "cybersecurity_attacks_linear_svm_evaluation.json").exists()), None)
if models_dir is None:
    raise FileNotFoundError("Não encontrei o JSON de avaliação em models/. Confirma a pasta do projeto.")

with (models_dir / "cybersecurity_attacks_linear_svm_evaluation.json").open(encoding="utf-8") as f:
    metrics = json.load(f)
print("Ficheiro carregado:", models_dir / "cybersecurity_attacks_linear_svm_evaluation.json")
print("Amostras de teste:", metrics.get("n_test", "não indicado"))
print("Categorias:", len(metrics.get("classes", [])))


## 2. Métricas globais

**Accuracy** é a proporção de previsões corretas. **Precision macro**, **recall macro** e **F1 macro** atribuem o mesmo peso a cada categoria, mesmo que tenham quantidades diferentes de exemplos.

In [ ]:
metric_names = {
    "accuracy": "Accuracy",
    "precision_macro": "Precision macro",
    "recall_macro": "Recall macro",
    "f1_macro": "F1-score macro",
    "f1_weighted": "F1-score weighted",
}
summary = pd.DataFrame([
    {"Métrica": label, "Valor (%)": round(metrics[key] * 100, 2)}
    for key, label in metric_names.items() if metrics.get(key) is not None
])
display(summary)

ax = summary.plot.bar(x="Métrica", y="Valor (%)", legend=False, figsize=(9, 4), rot=25)
ax.set_ylabel("Percentagem (%)")
ax.set_title("Desempenho do Linear SVM no conjunto de teste")
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()


## 3. Desempenho por categoria

O F1-score por categoria ajuda a identificar onde o classificador funciona melhor e pior. Valores elevados em classes específicas não garantem o mesmo desempenho em cenários reais.

In [ ]:
report = metrics.get("classification_report", {})
rows = []
for category, values in report.items():
    if isinstance(values, dict) and "f1-score" in values and category not in ("macro avg", "weighted avg", "micro avg"):
        rows.append({
            "Categoria": category,
            "Precision": round(values.get("precision", 0), 3),
            "Recall": round(values.get("recall", 0), 3),
            "F1-score": round(values["f1-score"], 3),
            "Amostras": int(values.get("support", 0)),
        })
per_class = pd.DataFrame(rows)
if per_class.empty:
    print("Relatório por categoria não disponível no JSON.")
else:
    print("Categorias com menor F1-score:")
    display(per_class.sort_values("F1-score").head(10))
    print("Categorias com maior F1-score:")
    display(per_class.sort_values("F1-score", ascending=False).head(10))


## 4. Matriz de confusão

A diagonal representa previsões corretas. Os valores fora da diagonal mostram confusões entre categorias. Para 27 categorias, a matriz é extensa; podemos ampliar a imagem no VS Code.

In [ ]:
matrix_csv = models_dir / "cybersecurity_attacks_linear_svm_confusion_matrix.csv"
if matrix_csv.exists():
    cm = pd.read_csv(matrix_csv, index_col=0)
    print("Dimensões da matriz:", cm.shape)
    print("Total de exemplos na matriz:", int(cm.to_numpy().sum()))
    print("Previsões corretas (diagonal):", int(cm.to_numpy().trace()))
else:
    print("CSV da matriz não encontrado.")

matrix_png = models_dir / "cybersecurity_attacks_linear_svm_confusion_matrix.png"
if matrix_png.exists():
    display(Image(filename=str(matrix_png)))
else:
    print("Imagem da matriz não encontrada.")


## 5. ROC-AUC e interpretação

A ROC-AUC mede a capacidade de distinguir cada classe das restantes, em diferentes limiares. No caso do **LinearSVC**, o valor multiclasse foi calculado com **scores de decisão**, que **não são probabilidades calibradas**. Não se deve interpretar a ROC-AUC como percentagem de previsões corretas.

In [ ]:
auc_value = metrics.get("roc_auc_ovr_macro")
score_type = metrics.get("roc_score_type")
if auc_value is None:
    print("ROC-AUC não disponível neste ficheiro de avaliação.")
else:
    print(f"ROC-AUC OvR macro: {auc_value:.4f}")
    print("Tipo de score:", score_type or "não especificado")
    if score_type == "decision":
        print("Nota: scores de decisão não são probabilidades.")


## 6. Conclusões e limitações

- O **Linear SVM** foi escolhido com base na comparação por validação cruzada no conjunto de treino, não pelo resultado do teste.
- A avaliação final usa um conjunto de teste separado, com **1.934 exemplos** (confirmar na saída acima).
- O **F1 macro** é particularmente útil porque considera todas as categorias com o mesmo peso.
- A matriz de confusão e o relatório por classe permitem identificar categorias mais difíceis.
- Apesar dos bons resultados, o dataset pode conter padrões repetitivos ou artificiais; remover textos exatamente iguais não elimina todas as formas de semelhança ou fuga de informação.
- O desempenho obtido **não garante** a mesma qualidade em ataques reais ou dados de outras fontes.

**Próxima etapa:** demonstrar a inferência com descrições novas e integrar o classificador com os outros agentes do grupo.